In [ ]:
from abc import ABC, abstractmethod
import time
import os

def leer_entero(mensaje, minimo, maximo):
    while True:
        texto = input(mensaje).strip()
        if not texto.isdigit():
            print("Ingresa un número entero.")
        else:
            valor = int(texto)
            if valor >= minimo and valor <= maximo:
                return valor
            print(f"El valor debe estar entre {minimo} y {maximo}.")


def limpiar_pantalla():
    try:
        get_ipython()  
        from IPython.display import clear_output
        clear_output(wait=False)
    except NameError:
        os.system("cls" if os.name == "nt" else "clear")

class IVistaConfiguracion(ABC):
    @abstractmethod
    def mostrar_bienvenida(self):
        pass

    @abstractmethod
    def pedir_numero_jugadores(self, minimo, maximo):
        pass

    @abstractmethod
    def pedir_nombre_jugador(self, numero):
        pass

    @abstractmethod
    def pedir_categoria(self, categorias):
        pass

    @abstractmethod
    def pedir_num_rondas(self):
        pass

    @abstractmethod
    def pedir_tiempo_debate(self):
        pass

    @abstractmethod
    def pedir_ultima_oportunidad(self):
        pass

    @abstractmethod
    def pedir_longitud_maxima_pista(self):
        pass

    @abstractmethod
    def mostrar_error(self, mensaje):
        pass


class IVistaRonda(ABC):
    @abstractmethod
    def mostrar_inicio_ronda(self, ronda, jugadores_activos):
        pass

    @abstractmethod
    def pedir_pista(self, jugador, numero_turno):
        pass

    @abstractmethod
    def mostrar_pistas(self, pistas):
        pass

    @abstractmethod
    def esperar_debate(self, segundos):
        pass

    @abstractmethod
    def pedir_voto(self, votante, candidatos):
        pass

    @abstractmethod
    def mostrar_resultado_votacion(self, eliminado):
        pass

    @abstractmethod
    def mostrar_error(self, mensaje):
        pass


class IVistaPartida(ABC):
    @abstractmethod
    def mostrar_rol_privado(self, jugador, categoria):
        pass

    @abstractmethod
    def pedir_intento_impostor(self, impostor):
        pass

    @abstractmethod
    def mostrar_resultado_final(self, resultado):
        pass

    @abstractmethod
    def mostrar_error(self, mensaje):
        pass


class VistaConfiguracionConsola(IVistaConfiguracion):
    def mostrar_bienvenida(self):
        print("            EL IMPOSTOR")

    def pedir_numero_jugadores(self, minimo, maximo):
        return leer_entero(f"Número de jugadores ({minimo}-{maximo}): ", minimo, maximo)

    def pedir_nombre_jugador(self, numero):
        return input(f"Nombre del jugador {numero}: ").strip()

    def pedir_categoria(self, categorias):
        print("\nCategorías disponibles:")
        numero = 1
        for nombre in categorias:
            print(f"  {numero}. {nombre}")
            numero = numero + 1
        opcion = leer_entero("Elige una categoría: ", 1, len(categorias))
        return categorias[opcion - 1]

    def pedir_num_rondas(self):
        return leer_entero("Número de rondas (1-10): ", 1, 10)

    def pedir_tiempo_debate(self):
        return leer_entero("Tiempo de debate en segundos (6-600): ", 6, 600)

    def pedir_ultima_oportunidad(self):
        while True:
            respuesta = input("¿El impostor tendra una ultima oportunidad? (s/n): ").strip().lower()
            if respuesta in ("s", "si", "sí"):
                return True
            if respuesta in ("n", "no"):
                return False
            print("  Responde 's' o 'n'.")

    def pedir_longitud_maxima_pista(self):
        return leer_entero("Longitud máxima de una pista (5-100): ", 5, 100)
            

    def mostrar_error(self, mensaje):
        print(f" {mensaje}")


class VistaRondaConsola(IVistaRonda):
    def __init__(self, esperar_debate=True):
        self.esperar = esperar_debate

    def mostrar_inicio_ronda(self, ronda, jugadores_activos):
        print("\n" + "-" * 50)
        print(f"  RONDA {ronda.numero}  -  Fase: {ronda.fase.value}")
        print("-" * 50)
        nombres = []
        for jugador in jugadores_activos:
            nombres.append(jugador.nombre)
        print("Jugadores activos: " + ", ".join(nombres))

    def pedir_pista(self, jugador, numero_turno):
        return input(f"\nTurno {numero_turno} - {jugador.nombre}, escribe tu pista: ").strip()

    def mostrar_pistas(self, pistas):
        print("\n Pistas de la ronda")
        for pista in pistas:
            print(f"  {pista.numero_turno}. {pista.get_autor().nombre}: {pista.get_texto()}")

    def esperar_debate(self, segundos):
        if self.esperar:
            try:
                for restante in range(segundos, 0, -1):
                    print(f"\r  Quedan {restante} s ", end="")
                    time.sleep(1)
            except KeyboardInterrupt:
                pass
            print("\n  Debate terminado.")

    def pedir_voto(self, votante, candidatos):
        print(f"\n{votante.nombre}, ¿a quién votas?")
        numero = 1
        for candidato in candidatos:
            print(f"  {numero}. {candidato.nombre}")
            numero = numero + 1
        opcion = leer_entero("Tu voto: ", 1, len(candidatos))
        limpiar_pantalla()
        return candidatos[opcion - 1]

    def mostrar_resultado_votacion(self, eliminado):
        print("\n RESULTADO DE LA VOTACIÓN")
        if eliminado is None:
            print("  Empate: nadie es eliminado en esta ronda.")
        else:
            print(f"  {eliminado.nombre} ha sido eliminado.")

    def mostrar_error(self, mensaje):
        print(f" {mensaje}")


class VistaPartidaConsola(IVistaPartida):
    def mostrar_rol_privado(self, jugador, categoria):
        input(f"\nPasa el dispositivo a {jugador.nombre}. Nadie más debe mirar. Presiona Enter para ver tu rol...")
        rol = jugador.get_rol()
        print(f"  Rol:       {rol.nombre}")
        print(f"  Categoría: {categoria}")
        print(f"  Palabra:   {rol.obtener_palabra_visible()}")
        input("Presiona Enter para ocultar tu rol...")
        limpiar_pantalla()

    def pedir_intento_impostor(self, impostor):
        print(f"\n{impostor.nombre} era el IMPOSTOR. ¡Tiene una última oportunidad!")
        return input("¿Cuál crees que es la palabra secreta?: ").strip()

    def mostrar_resultado_final(self, resultado):
        print(f"  GANAN: {resultado.ganador.value.upper()}")
        print(f"  Motivo:   {resultado.motivo}")
        print(f"  Palabra:  {resultado.palabra_secreta.texto}")
        print(f"  Impostor: {resultado.impostor.nombre}")
        print("\n  Roles de la partida:")
        for jugador in resultado.participantes:
            estado = "  (ganó)" if jugador in resultado.ganadores else ""
            print(f" {jugador.nombre}: {jugador.get_rol().nombre}{estado}")
    
    def mostrar_error(self, mensaje):
            print(f" {mensaje}")